# Day 8 — MLP from scratch (numpy) + PyTorch on MNIST


## 1. 2-layer MLP from scratch

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# XOR-like data
X = rng.uniform(-1, 1, (500, 2))
y = (X[:,0] * X[:,1] > 0).astype(int)  # checkerboard

def sigmoid(z): return 1/(1+np.exp(-z))
def relu(z): return np.maximum(0, z)

H = 16
W1 = rng.normal(0, 0.5, (2, H));  b1 = np.zeros(H)
W2 = rng.normal(0, 0.5, (H, 1));  b2 = np.zeros(1)

lr = 0.05
for epoch in range(3000):
    z1 = X @ W1 + b1; a1 = relu(z1)
    z2 = a1 @ W2 + b2; p = sigmoid(z2).squeeze()
    loss = -np.mean(y*np.log(p+1e-9) + (1-y)*np.log(1-p+1e-9))
    # backward
    dz2 = (p - y).reshape(-1,1) / len(X)
    dW2 = a1.T @ dz2; db2 = dz2.sum(0)
    da1 = dz2 @ W2.T
    dz1 = da1 * (z1 > 0)
    dW1 = X.T @ dz1; db1 = dz1.sum(0)
    W1 -= lr*dW1; b1 -= lr*db1
    W2 -= lr*dW2; b2 -= lr*db2
    if epoch % 500 == 0:
        acc = ((p > 0.5) == y).mean()
        print(f'ep {epoch:4d} loss={loss:.3f} acc={acc:.3f}')


## 2. PyTorch MLP on MNIST

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST('./data', train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=tfm)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=512)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),  nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x): return self.net(x)

model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step()
    # eval
    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(1)
            correct += (pred == yb).sum().item(); total += len(yb)
    print(f'epoch {epoch}: test acc = {correct/total:.4f}')
